# TaskMaster-VN — Fine-tune SLM tiếng Việt Agentic
## Hai chế độ
- `CPU_MODE = True` (mặc định): test toàn bộ pipeline trên CPU với Qwen2.5-0.5B, ~5–15 phút, không cần GPU.
- `CPU_MODE = False`: train thật Qwen2.5-7B QLoRA trên RTX 4060 32GB, ~8–20 giờ.

## Quy trình
1. Chạy hết notebook với `CPU_MODE = True` để verify.
2. Đổi `CPU_MODE = False` vào buổi tối khi có GPU.
3. Set `HF_TOKEN` environment variable để push model lên Hugging Face.
4. Chạy lại từ Cell 5.


In [ ]:
import os, sys, platform, time, json
import torch

# ============ ĐỔI THÀNH False KHI CHẠY TRÊN RTX 4060 ============
CPU_MODE = True
# ===============================================================

print(f"Python: {platform.python_version()}")
print(f"CPU_MODE: {CPU_MODE}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram:.1f} GB")
else:
    vram = 0

if not CPU_MODE and not torch.cuda.is_available():
    raise RuntimeError("CPU_MODE=False nhưng không có GPU. Đổi lại CPU_MODE=True.")

# Chọn base model theo VRAM
if CPU_MODE:
    BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"          # test pipeline
elif vram >= 24:
    BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct"          # 32GB dư sức
elif vram >= 16:
    BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"
else:
    BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct"
print(f"Base model: {BASE_MODEL}")


In [ ]:
# Cell này optional — nếu đã pip install -r requirements.txt thì bỏ qua
import subprocess
def sh(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=False)

# CPU test: deps đã cài sẵn qua requirements.txt → skip pip install
# (tránh build llama-cpp-python từ source trên Windows gây treo notebook)
print("skip pip install (CPU mode) — deps đã có sẵn")
if not CPU_MODE:
    sh('pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')
    sh("pip install -q --no-deps trl peft bitsandbytes")


In [ ]:
from huggingface_hub import list_datasets

KEYWORDS = [
    "vietnamese instruct", "VietnamAIHub", "5CD-AI",
    "glaive function calling", "agentic", "tool use",
]

seen = set()
print("=== Dataset khớp keyword (top 15 mỗi keyword) ===\n")
for kw in KEYWORDS:
    print(f"--- Keyword: {kw} ---")
    try:
        for ds in list_datasets(search=kw, limit=15):
            if ds.id in seen: continue
            seen.add(ds.id)
            print(f"  {ds.id}  | downloads={getattr(ds,'downloads',0)}")
    except Exception as e:
        print(f"  Lỗi: {e}")
    print()


In [ ]:
from datasets import load_dataset, concatenate_datasets, Dataset
import re

LIMIT = 2000 if CPU_MODE else None   # CPU test: 2000 mẫu; GPU: full

def to_chatml(ex):
    instr = ex.get("instruction") or ex.get("prompt") or ex.get("question") or ""
    out = ex.get("output") or ex.get("response") or ex.get("answer") or ""
    if not instr or not out:
        return {"text": ""}
    text = f"<|im_start|>user\n{instr}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>"
    return {"text": text}

def try_load(name, split="train"):
    try:
        ds = load_dataset(name, split=split)
        if LIMIT: ds = ds.select(range(min(LIMIT, len(ds))))
        print(f"✓ {name}: {len(ds)} mẫu")
        return ds
    except Exception as e:
        print(f"✗ {name}: {e}")
        return None

def clean(ds):
    # Đưa từng dataset về cột text TRƯỚC khi ghép (tránh lệch schema giữa các dataset)
    ds = ds.map(to_chatml, remove_columns=ds.column_names)
    ds = ds.filter(lambda x: 10 < len(x["text"]) < 4096)
    return ds

parts = []
for name in [
    "VietnamAIHub/General_Domain_Vietnamese_Instructions_V1",
    "glaiveai/glaive-function-calling-v2",
    "II-Vietnam/Agentic-Search-MultiHop",
]:
    ds = try_load(name)
    if ds is not None:
        ds = clean(ds)
        if len(ds) > 0:
            parts.append(ds)

if not parts:
    print("Không load được dataset usable nào — tạo 20 mẫu giả lập để test pipeline.")
    fake = [
        {"instruction": f"Giao task mẫu số {i}: tóm tắt email và tạo báo cáo",
         "output": f"Kế hoạch: 1) Đọc email 2) Tóm tắt 3) Xuất Excel. Xong task {i}."}
        for i in range(20)
    ]
    parts = [clean(Dataset.from_list(fake))]

ds = concatenate_datasets(parts) if len(parts) > 1 else parts[0]

# Khử trùng lặp
seen = set(); keep = []
for i, t in enumerate(ds["text"]):
    if t not in seen:
        seen.add(t); keep.append(i)
ds = ds.select(keep)

print(f"\n=== Tổng mẫu sau clean: {len(ds)} ===")
os.makedirs("../data", exist_ok=True)
ds.to_json("../data/train_chatml.jsonl", force_ascii=False)
print("Đã lưu ../data/train_chatml.jsonl")


In [ ]:
if CPU_MODE:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    print(f"[CPU MODE] Load {BASE_MODEL} (float32)")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
    print("✓ Load xong (test inference)")
else:
    from unsloth import FastLanguageModel
    print(f"[GPU MODE] Load {BASE_MODEL} QLoRA 4-bit")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=2048,
        load_in_4bit=True,
        dtype=None,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=32, lora_alpha=32, lora_dropout=0.0,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        use_gradient_checkpointing="unsloth",
    )
    print("✓ QLoRA adapter xong")


In [ ]:
if CPU_MODE:
    # Test pipeline: 2 prompt tiếng Việt
    prompts = [
        "Liệt kê 3 bước xử lý file Excel 10.000 dòng.",
        "Giao task: tóm tắt 50 email và tạo báo cáo. Chia nhỏ các bước.",
    ]
    for p in prompts:
        msgs = [{"role":"user","content":p}]
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt")
        t0 = time.time()
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
        dt = time.time() - t0
        ans = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        print(f"\nPROMPT: {p}")
        print(f"OUTPUT: {ans}")
        print(f"({dt:.1f}s)\n" + "-"*60)
else:
    from datasets import load_dataset
    from trl import SFTTrainer
    from transformers import TrainingArguments

    train_ds = load_dataset("json", data_files="../data/train_chatml.jsonl", split="train")
    print(f"Train samples: {len(train_ds)}")

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=train_ds,
        dataset_text_field="text", max_seq_length=2048,
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=10, max_steps=500, learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10, output_dir="../outputs",
            optim="adamw_8bit", save_steps=100,
        ),
    )
    trainer.train()

    model.save_pretrained_merged("../models/merged_16bit", tokenizer, save_method="merged_16bit")
    model.save_pretrained_gguf("../models/gguf", tokenizer, quantization_method="q4_k_m")
    print("✓ Đã merge + export GGUF")


In [ ]:
if not CPU_MODE:
    from huggingface_hub import login
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        print("⚠ Thiếu HF_TOKEN — bỏ qua push model.")
    else:
        login(token=hf_token)
        repo = os.environ.get("HF_REPO", "YOUR_USERNAME/TaskMaster-VN-GGUF")
        model.push_to_hub_gguf(repo, tokenizer, quantization_method="q4_k_m")
        print(f"✓ Đã push: https://huggingface.co/{repo}")
else:
    print("[CPU MODE] Bỏ qua push model. Chạy lại với CPU_MODE=False khi có GPU.")


In [ ]:
print("="*60)
print("TỔNG KẾT")
print("="*60)
print(f"CPU_MODE: {CPU_MODE}")
print(f"Base model: {BASE_MODEL}")
print(f"Dataset: ../data/train_chatml.jsonl")
print("\nChecklist khi bật GPU RTX 4060:")
print("  [ ] Đổi CPU_MODE = False ở Cell 1")
print("  [ ] Set os.environ['HF_TOKEN'] = 'hf_...'")
print("  [ ] Set os.environ['HF_REPO'] = 'username/TaskMaster-VN-GGUF'")
print("  [ ] Chạy lại Cell 5 → 7")
print("  [ ] Cập nhật app.py dùng model mới")
